# 04 — H&M: gói dữ liệu phục vụ web (serving assets)

Notebook này **không train gì**. Nó xuất những thứ web/RAG cần mà notebook 00–03 chưa xuất:

| Đầu ra | Dùng cho |
|---|---|
| `item_catalog.parquet` | Toàn bộ 25 cột `articles.csv` + thống kê bán đến mốc `AS_OF` (bán 7/28 ngày, ngày đầu/cuối, giá trung vị, tuổi TB người mua). Thứ tự dòng = `items.parquet` (`product_id = dòng + 1`) |
| `personas.parquet`, `personas_history.parquet`, `personas_heldout.parquet` | Tài khoản demo lấy từ khách thật trong tập test: hồ sơ, lịch sử mua trước `AS_OF`, đơn mua thật trong tuần test |
| `hm_images_rest_part01.zip` | Ảnh (512 px, JPEG 85) của các item **không** có trong gói `active_only` đã tải trước đó (chủ yếu item cold) |
| `serving_manifest.json` | Số lượng, mốc thời gian, kích thước file |

**Cách chạy (Kaggle, CPU là đủ):** Add Input = cuộc thi *H&M Personalized Fashion Recommendations* **và** Dataset `hm-dataset-filled` → bật Internet nếu cần → *Save Version → Save & Run All*. Lấy file ở tab Output.

Mốc thời gian `AS_OF = 2020-09-16` (cutoff của cửa sổ test): thống kê chỉ dùng giao dịch **trước** ngày này, đúng cấu hình đã đo trong luận văn.

In [ ]:
import os, json, datetime as dt, time
from pathlib import Path
import numpy as np
import polars as pl

AS_OF = dt.date.fromisoformat(os.getenv("HM_AS_OF", "2020-09-16"))   # đồng hồ của shop = cutoff test
SEED = 20261006
N_PERSONAS = int(os.getenv("HM_N_PERSONAS", "30"))
INPUT_ROOT = Path(os.getenv("HM_INPUT_ROOT", "/kaggle/input"))
OUT = Path(os.getenv("HM_SERVING_DIR", "/kaggle/working/hm_serving")); OUT.mkdir(parents=True, exist_ok=True)


def find(name, want_dir=False):
    # Kaggle mount: /kaggle/input/<slug>/x hoặc /kaggle/input/datasets/<owner>/<slug>/x -> thử độ sâu 0..3
    for depth in range(0, 4):
        pattern = "/".join(["*"] * depth + [name])
        for p in INPUT_ROOT.glob(pattern):
            if (p.is_dir() if want_dir else p.is_file()):
                return p
    raise FileNotFoundError(f"Không thấy {name} dưới {INPUT_ROOT}. Đã Add Input chưa?")


TRANSACTIONS = find("transactions_train.csv")
ARTICLES = find("articles.csv")
CUSTOMERS = find("customers.csv")
IMAGE_DIR = find("images", want_dir=True)
DATASET_DIR = find("items.parquet").parent          # hm-dataset-filled
for n, p in dict(transactions=TRANSACTIONS, articles=ARTICLES, customers=CUSTOMERS, images=IMAGE_DIR, dataset=DATASET_DIR).items():
    print(f"{n:<13}{p}")
print("AS_OF =", AS_OF)

## 1. Catalog + thống kê bán đến `AS_OF`

Giữ **nguyên thứ tự** `items.parquet` (đã kiểm: trùng `item_ids` trong checkpoint tower, `product_id = dòng + 1`). Giá trong dữ liệu H&M đã bị chuẩn hoá (trung vị ≈ 0,025) nên chỉ lưu giá thô; việc quy đổi sang VND làm ở bước import.

In [ ]:
t0 = time.time()
items = pl.read_parquet(DATASET_DIR / "items.parquet", columns=["item_id"])
articles = (pl.read_csv(ARTICLES, schema_overrides={"article_id": pl.Utf8})
            .with_columns(pl.col("article_id").str.zfill(10)))
assert articles.height == items.height, (articles.height, items.height)

customers = (pl.scan_csv(CUSTOMERS, schema_overrides={"customer_id": pl.Utf8})
             .select("customer_id", "age", "club_member_status", "fashion_news_frequency"))

d7, d28 = AS_OF - dt.timedelta(days=7), AS_OF - dt.timedelta(days=28)
tx = (pl.scan_csv(TRANSACTIONS, try_parse_dates=True,
                  schema_overrides={"article_id": pl.Utf8, "customer_id": pl.Utf8})
      .select("t_dat", "customer_id", "article_id", "price", "sales_channel_id")
      .filter(pl.col("t_dat") < AS_OF)
      .with_columns(pl.col("article_id").str.zfill(10))
      .join(customers.select("customer_id", "age"), on="customer_id", how="left"))

stats = (tx.group_by("article_id").agg(
            pl.len().alias("n_all"),
            (pl.col("t_dat") >= d7).sum().alias("n_7d"),
            (pl.col("t_dat") >= d28).sum().alias("n_28d"),
            pl.col("t_dat").min().alias("first_seen"),
            pl.col("t_dat").max().alias("last_seen"),
            pl.col("price").median().alias("price_median"),
            pl.col("price").mean().alias("price_mean"),
            (pl.col("sales_channel_id") == 2).mean().alias("online_share"),
            pl.col("age").mean().alias("buyer_age_mean"))
         .collect(streaming=True))
print("stats:", stats.shape, f"{time.time() - t0:.0f}s")

catalog = (items.join(articles, left_on="item_id", right_on="article_id", how="left")
           .join(stats, left_on="item_id", right_on="article_id", how="left")
           .with_columns(pl.col(["n_all", "n_7d", "n_28d"]).fill_null(0).cast(pl.Int64))
           .with_row_index("product_id", offset=1))
has_image_src = [(IMAGE_DIR / i[:3] / f"{i}.jpg").is_file() for i in catalog["item_id"].to_list()]
catalog = catalog.with_columns(pl.Series("has_image_src", has_image_src))

assert catalog.height == 105_542 and catalog["prod_name"].null_count() == 0
assert catalog["product_id"].min() == 1 and catalog["item_id"].to_list() == items["item_id"].to_list()
catalog.write_parquet(OUT / "item_catalog.parquet", compression="zstd")

print("columns:", catalog.columns)
print("giao dịch < AS_OF:", int(catalog["n_all"].sum()))
print("never-sold (n_all == 0, thống kê toàn cục):", int((catalog["n_all"] == 0).sum()), "  (33.013 trong tài liệu là item cold theo TOWER, degree = 0 trong mẫu 50k; số toàn cục nhỏ hơn nhiều)")
print("đang bán (n_28d > 0):  ", int((catalog["n_28d"] > 0).sum()))
print("có ảnh nguồn:", int(catalog["has_image_src"].sum()), "/", catalog.height)

## 2. Tài khoản demo từ khách thật trong tập test

Chọn ngẫu nhiên (seed cố định) khách có 5–40 giao dịch trước `AS_OF` và ít nhất 1 món mua trong tuần test. `personas_heldout` là đáp án thật của tuần 16–22/09/2020: dùng để minh hoạ gợi ý cá nhân hoá và đối chiếu định lượng, **không** đưa vào lịch sử demo.

In [ ]:
def load_split(name):
    return pl.read_parquet(DATASET_DIR / f"{name}.parquet", columns=["customer_id", "article_id", "t_dat"]) \
             .with_columns(pl.col("article_id").cast(pl.Utf8).str.zfill(10), pl.col("t_dat").cast(pl.Date))

history_all = pl.concat([load_split(n) for n in ("train", "rerank_train", "valid")])
heldout_all = load_split("test")
assert history_all["t_dat"].max() < AS_OF <= heldout_all["t_dat"].min(), (history_all["t_dat"].max(), heldout_all["t_dat"].min())

hist_n = history_all.group_by("customer_id").len().rename({"len": "n_hist"})
eligible = (heldout_all.select("customer_id").unique().join(hist_n, on="customer_id")
            .filter(pl.col("n_hist").is_between(5, 40)).sort("customer_id"))
rng = np.random.RandomState(SEED)
chosen = eligible["customer_id"].to_numpy()
chosen = sorted(rng.choice(chosen, size=min(N_PERSONAS, len(chosen)), replace=False).tolist())
print(f"đủ điều kiện: {eligible.height:,} khách -> chọn {len(chosen)}")

profile = (customers.filter(pl.col("customer_id").is_in(chosen)).collect()
           .join(hist_n, on="customer_id").sort("customer_id"))
hist = history_all.filter(pl.col("customer_id").is_in(chosen)).sort(["customer_id", "t_dat", "article_id"])
held = heldout_all.filter(pl.col("customer_id").is_in(chosen)).sort(["customer_id", "t_dat", "article_id"])
profile.write_parquet(OUT / "personas.parquet")
hist.write_parquet(OUT / "personas_history.parquet")
held.write_parquet(OUT / "personas_heldout.parquet")
print(profile.head(5)); print("history rows:", hist.height, "| heldout rows:", held.height)

## 3. Ảnh còn thiếu (ngoài gói `active_only`)

Gói `active_only` chỉ gồm item mà 50k khách mẫu từng mua (≈ 73k). Các item còn lại — bán cho khách ngoài mẫu hoặc chưa từng bán, tức đúng nhóm mà nguồn `recent`, `new` và kênh cold có thể đưa lên — chưa có ảnh trên máy. Ô này đóng gói phần còn lại bằng đúng cách resize của `export_hm_images_zip.ipynb` (512 px, JPEG 85) để dùng chung một thư mục ảnh.

In [ ]:
import io, zipfile
from concurrent.futures import ThreadPoolExecutor
from PIL import Image

active = set()
for s in ("train", "rerank_train", "valid", "test"):
    col = pl.read_parquet(DATASET_DIR / f"{s}.parquet", columns=["article_id"])["article_id"]
    active.update(col.cast(pl.Utf8).str.zfill(10).unique().to_list())
rest = [i for i, ok in zip(catalog["item_id"].to_list(), catalog["has_image_src"].to_list()) if ok and i not in active]
print(f"active (đã có ảnh trong gói cũ): {len(active):,} | cần xuất thêm: {len(rest):,}")


def prep(item):
    try:
        with Image.open(IMAGE_DIR / item[:3] / f"{item}.jpg") as im:
            im = im.convert("RGB")
            if max(im.size) > 512:
                s = 512 / max(im.size)
                im = im.resize((int(im.width * s), int(im.height * s)), Image.Resampling.LANCZOS)
            buf = io.BytesIO(); im.save(buf, format="JPEG", quality=85, optimize=True)
            return item, buf.getvalue(), None
    except Exception as e:  # noqa: BLE001
        return item, None, str(e)


zip_path = OUT / "hm_images_rest_part01.zip"
ok = bad = 0
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_STORED) as zf, ThreadPoolExecutor(max_workers=8) as ex:
    for item, data, err in ex.map(prep, rest):
        if data is None:
            bad += 1
        else:
            zf.writestr(f"images/{item[:3]}/{item}.jpg", data); ok += 1
print(f"zip: {ok:,} ảnh, lỗi {bad}, {zip_path.stat().st_size / 1e6:.0f} MB")

In [ ]:
manifest = {
    "as_of": AS_OF.isoformat(), "seed": SEED, "items": catalog.height,
    "never_sold": int((catalog["n_all"] == 0).sum()), "selling_28d": int((catalog["n_28d"] > 0).sum()),
    "personas": len(chosen), "images_rest": ok, "images_rest_failed": bad, "images_active_expected_in_old_zip": len(active),
    "files": {p.name: round(p.stat().st_size / 1e6, 2) for p in sorted(OUT.iterdir()) if p.name != "serving_manifest.json"},
}
(OUT / "serving_manifest.json").write_text(json.dumps(manifest, indent=2))
print(json.dumps(manifest, indent=2))